# Notebook hướng dẫn Module TV7

> Notebook này được chuyển đổi từ `HUONG_DAN_TV7.md` để có thể xem trực tiếp trên GitHub hoặc mở bằng Jupyter/Google Colab.

**Lưu ý:** Các cell lệnh cài đặt/chạy chương trình cần được thực thi tại thư mục gốc của dự án `traffic-congestion-cv`.


# BÁO CÁO KỸ THUẬT & HƯỚNG DẪN VẬN HÀNH MODULE TV7
**Dự án:** Hệ thống Giám sát & Đánh giá Mức độ Ùn tắc Giao thông Đô thị từ Camera quan sát (UTH)  
**Học phần:** Xử lý ảnh và Thị giác máy tính (121036)  
**Thành viên phụ trách:** TV7  
**File mã nguồn chính:** `demo/app.py` | **File kiểm thử:** `test_tv7.py`

---

---

## 1. TỔNG QUAN VAI TRÒ VÀ NHIỆM VỤ CỦA TV7

Trong dự án, **TV7** giữ vai trò **Kỹ sư Tích hợp Hệ thống và Xây dựng Giao diện Giám sát (Dashboard & Demonstration)**. 

Nếu các thành viên từ TV1 đến TV6 chịu trách nhiệm nghiên cứu và phát triển từng thuật toán xử lý riêng lẻ:
- **TV2:** Tiền xử lý ảnh (Grayscale, Gaussian Blur, CLAHE) và Nắn phối cảnh Bird's-Eye View (BEV).
- **TV3:** Thuật toán trừ nền thích ứng (MOG2) và trích xuất mặt nạ chiếm dụng mặt đường (Occupancy Mask).
- **TV4:** Ước lượng vector chuyển động và tính vận tốc dòng xe qua Optical Flow (Farneback).
- **TV5:** Nhận dạng, phân loại phương tiện (YOLOv8) và quy đổi tải trọng tương đương (PCU).
- **TV1:** Kiến trúc Pipeline tổng thể và công thức đánh giá chỉ số ùn tắc TCI.

Thì nhiệm vụ cốt lõi của **TV7** là:
> **Tích hợp đồng bộ toàn bộ kết quả của 5 module trên vào một giao diện Web Dashboard tương tác thời gian thực**, đảm bảo hiển thị trực quan đồng thời 4 cửa sổ góc nhìn kỹ thuật, bảng đo lường chỉ số trung tâm và cho phép người dùng kiểm thử, tinh chỉnh tham số của toàn bộ hệ sinh thái thuật toán.

---

## 2. CHI TIẾT GIẢI PHÁP KỸ THUẬT VÀ BÀI LÀM CỦA TV7

### 2.1. Bộ điều phối luồng dữ liệu lõi (`DashboardPipeline`)
TV7 xây dựng class `DashboardPipeline` trong file `demo/app.py` với nhiệm vụ nạp luồng video (từ file mp4, webcam hoặc tệp tải lên) và điều phối xử lý tuần tự qua các module:

1. **Khởi tạo và cấu hình:**
   - Kết nối với bảng toạ độ thực nghiệm `VIDEO_CONFIG` (`configs/toadovideo.py`) để tự động nhận diện góc nắn phối cảnh chuẩn theo từng video.
   - Khởi tạo đồng thời: `RoadSegmenter` (TV3), `MotionEstimator` (TV4), `VehicleDetector` (TV5) và `TrafficCongestionEvaluator` (TV1).
2. **Xử lý từng khung hình (`process_frame`):**
   - Đo đạc chỉ số FPS thời gian thực dựa trên sai phân thời gian giữa các frame.
   - Gửi frame qua `preprocess_frame` và `get_perspective_bev` để sinh ảnh BEV.
   - Truyền ảnh tăng cường vào `segmenter.extract_occupancy` để lấy `occupancy_ratio` và mặt nạ nhị phân.
   - Ước lượng trường vector vận tốc qua `motion_est.estimate_speed`, lọc nhiễu và áp dụng bảng màu `cv2.COLORMAP_JET` để tạo Heatmap.
   - Nhận diện phương tiện, vẽ bounding box và tính tổng tải trọng `total_pcu` qua `detector.detect_and_count_pcu`.
   - Tổng hợp 3 chỉ số thành phần vào `evaluator.compute_tci` để tính TCI tức thời, TCI làm mịn và phân cấp mức độ ùn tắc.
3. **Cơ chế lặp vô tận (Auto-looping):**
   - Khi luồng video đọc đến frame cuối cùng, pipeline tự động reset con trỏ frame về `0` (`cap.set(cv2.CAP_PROP_POS_FRAMES, 0)`), giúp quá trình demo chạy liên tục không bị ngắt quãng.

---

### 2.2. Kiến trúc 4 Cửa sổ hiển thị (Grid 2x2 đồng bộ)
TV7 bố trí 4 góc nhìn kỹ thuật theo dạng lưới 2x2 với kích thước hiển thị được **chuẩn hóa đồng bộ 16:9 (`640x360`)**:


```
+------------------------------------+------------------------------------+
|  CỬA SỔ 1: Camera gốc & BBox (TV5) |  CỬA SỔ 2: Bird's-Eye View (TV2)   |
+------------------------------------+------------------------------------+
|  CỬA SỔ 3: Mặt nạ chiếm dụng (TV3) |  CỬA SỔ 4: Heatmap vận tốc (TV4)   |
+------------------------------------+------------------------------------+
```


- **Cửa sổ 1 - Khung hình gốc & Nhận diện xe (TV5):**
  Hiển thị trực tiếp video từ camera với các bounding box phân loại màu theo 4 nhóm phương tiện (xe máy: xanh cyan, ô tô: xanh lá, xe buýt: cam, xe tải: đỏ) kèm nhãn và độ tin cậy.
- **Cửa sổ 2 - Nắn phối cảnh Bird's-Eye View (TV2):**
  Hiển thị mặt đường đã được nắn phối cảnh thẳng từ trên cao xuống thông qua phép biến đổi ma trận `warpPerspective`, loại bỏ hoàn toàn hiện tượng biến dạng xa gần.
- **Cửa sổ 3 - Mặt nạ chiếm dụng mặt đường (TV3):**
  Mặt nạ nhị phân đen/trắng biểu diễn chính xác diện tích mặt đường bị phương tiện che phủ sau khi đã lọc bóng râm và khử nhiễu hình thái học.
- **Cửa sổ 4 - Bản đồ nhiệt vận tốc dòng xe (TV4):**
  Bản đồ nhiệt màu JET thể hiện độ lớn vector chuyển động quang học (Optical Flow); vùng màu ấm (đỏ/vàng) thể hiện xe đang di chuyển nhanh, vùng màu lạnh (xanh dương) thể hiện xe dừng hoặc di chuyển chậm.

---

### 2.3. Bảng điều khiển trung tâm (Central HUD Metrics)
Phía trên 4 cửa sổ là thanh hiển thị 6 thẻ chỉ số được thiết kế đồng bộ chiều cao (`68px`), thể hiện toàn bộ trạng thái giao thông tức thời:

| Thẻ chỉ số | Ý nghĩa kỹ thuật | Đơn vị / Định dạng |
| :--- | :--- | :--- |
| **TỐC ĐỘ** | Tốc độ xử lý khung hình của hệ thống | `FPS` (Frames Per Second) |
| **CHỈ SỐ TCI** | Chỉ số ùn tắc giao thông tổng hợp (TV1) | Thang đo từ `0.00` đến `1.00` |
| **CHIẾM DỤNG** | Tỷ lệ diện tích mặt đường bị xe chiếm chỗ (TV3) | Phần trăm `%` |
| **VẬN TỐC** | Tỷ lệ suy giảm/chuẩn hóa vận tốc dòng xe (TV4) | Phần trăm `%` so với tốc độ tự do |
| **TẢI TRỌNG PCU** | Tổng tải trọng quy đổi xe con tiêu chuẩn (TV5) | Điểm số `PCU` |
| **MỨC ĐỘ** | Nhãn phân cấp ùn tắc kèm viền màu trạng thái | 4 mức: Thông thoáng / Bình thường / Ùn ứ / Tắc nghẽn |

---

### 2.4. Các tối ưu hóa giao diện và trải nghiệm (UI/UX)
Nhằm mang lại trải nghiệm chuyên nghiệp, tinh tế, TV7 đã thực hiện các cải tiến kỹ thuật UI:
1. **Phong cách Dark Minimal:** Sử dụng bảng màu tối sang trọng (`#0f172a`, `#1e293b`), typography Inter hiện đại, loại bỏ hoàn toàn emoji rườm rà.
2. **Khoảng cách bố cục (Spacious Spacing):**
   - Thiết lập `padding-top: 4.5rem` để tiêu đề không bị thanh navbar của Streamlit che khuất.
   - Thêm khoảng đệm phân cách rõ ràng (`gap="large"` giữa các cột, `28px` giữa hàng trên và hàng dưới) tránh hiện tượng các khung video bị dính sát vào nhau.
3. **Hiệu ứng Skeleton Loading (Shimmer):**
   Khi người dùng thay đổi tham số hoặc đổi video, hệ thống tự động kích hoạt hiệu ứng quét sáng (shimmer animation) tại các ô chỉ số và khung video trước khi nạp frame mới, tạo cảm giác mượt mà như các ứng dụng web hiện đại.
4. **Chế độ tự động chạy (Instant Auto-play):**
   Mặc định khởi chạy ngay video mẫu đầu tiên khi vừa mở trang mà không cần bấm nút "Bắt đầu".

---

## 3. HƯỚNG DẪN CÀI ĐẶT VÀ KHỞI CHẠY HỆ THỐNG

### 3.1. Yêu cầu môi trường
- Hệ điều hành: Linux (Ubuntu/Zorin OS), macOS hoặc Windows.
- Python: Phiên bản từ `3.10` đến `3.12`.

### 3.2. Thiết lập môi trường ảo và cài đặt thư viện
Tại thư mục gốc của dự án (`traffic-congestion-cv`), mở terminal và chạy tuần tự các lệnh sau:


In [ ]:
# 1. Tạo môi trường ảo (venv)
!python3 -m venv venv

# 2. Kích hoạt môi trường ảo
# Trên Linux/macOS:
!source venv/bin/activate
# Trên Windows (Command Prompt):
# venv\Scripts\activate.bat
# Trên Windows (PowerShell):
# venv\Scripts\Activate.ps1

# 3. Nâng cấp pip và cài đặt toàn bộ thư viện cần thiết
!pip install --upgrade pip
!pip install -r requirements.txt


---

### 3.3. Khởi chạy Dashboard giao diện Web
Đảm bảo môi trường ảo đã được kích hoạt, chạy lệnh sau:


In [ ]:
!streamlit run demo/app.py


Sau khi chạy lệnh, trình duyệt web sẽ tự động mở trang dashboard tại địa chỉ:
👉 **`http://localhost:8501`**

*(Video mẫu `traffic_congested.mp4` sẽ tự động chạy ngay khi trang web được tải).*

---

### 3.4. Chạy kiểm thử tự động độc lập
Để kiểm tra tính toàn vẹn của pipeline TV7 mà không cần mở trình duyệt web:


In [ ]:
!python test_tv7.py


**Mục đích kiểm thử của `test_tv7.py`:**
- Xác minh tính tương thích và liên kết dữ liệu giữa cả 5 module.
- Kiểm tra tính hợp lệ của 4 khung hình (kích thước, định dạng 3 kênh BGR).
- Kiểm tra giá trị TCI tính ra nằm trong khoảng hợp lệ $[0.0, 1.0]$.
- Tự động xuất ảnh chụp nghiệm thu 4 cửa sổ tại: `data/processed/dashboard_tv7_snapshot.jpg`.

---

## 4. HƯỚNG DẪN THAO TÁC TRÊN GIAO DIỆN

Tại thanh điều khiển bên trái (**Sidebar**):
1. **Chọn nguồn video giám sát (`Nguồn video`):**
   - `Ùn tắc (traffic_congested.mp4)`: Video thực tế dòng xe đông đúc, dừng chờ tại TP.HCM.
   - `Thông thoáng (traffic_free_flow.mp4)`: Video cao tốc dòng xe lưu thông tốc độ cao.
   - `Đèn tín hiệu (traffic_traffic_light.mp4)`: Video ngã tư dừng chờ theo nhịp đèn.
   - `Tải lên tệp video...`: Tải lên video cá nhân định dạng `.mp4`, `.avi`, `.mov`.
   - `Webcam máy tính`: Sử dụng trực tiếp camera gắn trên máy tính để demo thời gian thực.
2. **Điều khiển luồng:**
   - Bấm nút **Tạm dừng** để đóng băng khung hình và phân tích chi tiết.
   - Bấm nút **Tiếp tục** để hệ thống chạy tiếp.
   - Thanh trượt **Bước nhảy frame**: Tăng lên 2 hoặc 3 để giảm tải CPU trên máy cấu hình yếu.
3. **Mục "Tham số nâng cao" (Expander):**
   Cho phép tinh chỉnh các tham số thuật toán trực tiếp khi đang chạy:
   - *Confidence / IoU NMS:* Thay đổi độ nhạy của bộ phát hiện YOLOv8 (TV5).
   - *Lọc nhiễu vận tốc:* Thay đổi ngưỡng lọc rung lắc camera của Optical Flow (TV4).
   - *MOG2 History / VarThreshold:* Thay đổi tốc độ học nền và độ nhạy trừ nền (TV3).
4. **Vùng đồ thị bên dưới:**
   - Biểu đồ đường thời gian thực theo dõi sự biến động của chỉ số TCI, tỷ lệ chiếm dụng và chuẩn hoá vận tốc.
   - Biểu đồ cột thống kê số lượng từng loại phương tiện đang có mặt trên khung hình.

---

## 5. KẾT QUẢ NGHIỆM THU VÀ KIỂM THỬ

### 5.1. Kết quả chạy kiểm thử `test_tv7.py`


```text
==================================================================
  KIEM THU TV7: DASHBOARD HIEN THI DEMO & KIEM THU
  Video kiem thu: data/raw/traffic_congested.mp4
==================================================================
  [Frame 01/25] TCI: 0.43 | Muc do: 2 - Binh thuong | Chiem dung: 0.0% | PCU: 7.2 | FPS: 7.8
  [Frame 05/25] TCI: 0.42 | Muc do: 2 - Binh thuong | Chiem dung: 0.0% | PCU: 7.2 | FPS: 5.6
  [Frame 10/25] TCI: 0.41 | Muc do: 2 - Binh thuong | Chiem dung: 0.0% | PCU: 7.2 | FPS: 4.0
  [Frame 15/25] TCI: 0.41 | Muc do: 2 - Binh thuong | Chiem dung: 0.0% | PCU: 7.2 | FPS: 3.4
  [Frame 20/25] TCI: 0.41 | Muc do: 2 - Binh thuong | Chiem dung: 0.3% | PCU: 7.2 | FPS: 3.1
  [Frame 25/25] TCI: 0.41 | Muc do: 2 - Binh thuong | Chiem dung: 0.1% | PCU: 7.2 | FPS: 2.3

[OK] Da xu ly thanh cong 25 frames (Toc do trung binh: 2.4 FPS)
[OK] Da xuat anh Canvas nghiem thu TV7 tai: data/processed/dashboard_tv7_snapshot.jpg
==================================================================
  KET QUA: MODULE TV7 DASHBOARD DA VUOT QUA TAT CA CAC KIEM THU!
==================================================================
```


### 5.2. File lưu trữ nghiệm thu
Ảnh chụp Canvas tích hợp đầy đủ 4 góc nhìn của TV7 được lưu tự động tại đường dẫn:
📁 `data/processed/dashboard_tv7_snapshot.jpg`
